In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 83.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 72.2 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:

import os
import socket
import threading
import time

import torch
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from pyngrok import conf, ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer

# ---- config --------------------------------------------------------------
MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"
API_KEY = os.environ.get("TRAVEL_API_KEY", "secret123") 
NGROK_TOKEN = "3H2XzFfa8bJt5yRhcREwNQgYtRl_35HrAoq6ixGwrsiHoFcC8"    










In [3]:
MAX_MEMORY = {0: "13GiB", 1: "13GiB", "cpu": "30GiB"}
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory=MAX_MEMORY,
    offload_folder="/kaggle/working/offload",
).eval()
 
 
def generate_text(prompt: str, max_length: int = 400) -> str:
    # IMPORTANT: Mistral-Nemo-Instruct only follows instructions reliably when
    # the prompt goes through its chat template. Feeding it a raw string makes
    # it behave more like a base model — it free-associates on whatever
    # pattern it's seen most, which is why it can drift onto the wrong
    # country/topic entirely.
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
 
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_length,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.7,
            do_sample=True,
        )
 
    # Decode only the newly generated tokens (skip the echoed prompt/template).
    new_tokens = outputs[0][input_ids.shape[-1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
 
    # Free fragmented memory between requests — helps avoid OOM creeping up
    # over a session with several back-to-back generations.
    del outputs, input_ids
    torch.cuda.empty_cache()
    return text

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
app = FastAPI(title="Travel Assistant LLM backend")
 
 
@app.post("/generate")
async def gen(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    data = await req.json()
    prompt = data.get("prompt", "")
    if not prompt:
        raise HTTPException(status_code=400, detail="Missing 'prompt'")
    try:
        return {"response": generate_text(prompt, data.get("max_length", 400))}
    except torch.cuda.OutOfMemoryError as e:
        torch.cuda.empty_cache()
        raise HTTPException(
            status_code=500,
            detail=f"CUDA OOM even after 4-bit quantization: {e}. Try a shorter prompt or lower max_length.",
        )
    except Exception as e:
        import traceback
        traceback.print_exc()  # shows the full stack trace in the Kaggle notebook output
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {e}")
 
 
@app.get("/health")
async def health():
    return {"status": "ok"}

In [5]:
# ---- ngrok tunnel + server thread ------------------------------------------
def free_port() -> int:
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port


port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)
print("Copy this URL + your TRAVEL_API_KEY into the Streamlit app's sidebar.")


def run():
    uvicorn.run(app, host="0.0.0.0", port=port)


threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://joyride-duty-hurler.ngrok-free.dev                                         
Copy this URL + your TRAVEL_API_KEY into the Streamlit app's sidebar.


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:42455 (Press CTRL+C to quit)
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


INFO:     196.131.160.162:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.131.160.162:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.131.160.162:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     196.131.160.162:0 - "POST /generate HTTP/1.1" 200 OK
